In [6]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Your user-defined preprocessing functions (included here for completeness)
def split_text(input_string):
    result = []
    word = ''
    for char in input_string:
        if char in (' ', '\t', '\n'):
            if word:
                result.append(word)
                word = ''
        else:
            word += char
    if word:
        result.append(word)
    return result

def remove_punctuation(words):
    punctuation_marks = ['.', ',', '!', '?', ';', ':', '"', "'", '(', ')',
                         '[', ']', '{', '}', '-', '_', '/', '\\', '`', '~',
                         '@', '#', '$', '%', '^', '&', '*', '=', '+', '<',
                         '>', '|']
    cleaned_words = []
    for word in words:
        clean_word = ''
        for char in word:
            if char not in punctuation_marks:
                clean_word += char
        if clean_word:
            cleaned_words.append(clean_word)
    return cleaned_words

def manual_lowercase(word):
    char_map = {
        'A': 'a', 'B': 'b', 'C': 'c', 'D': 'd', 'E': 'e', 'F': 'f', 'G': 'g', 'H': 'h', 'I': 'i',
        'J': 'j', 'K': 'k', 'L': 'l', 'M': 'm', 'N': 'n', 'O': 'o', 'P': 'p', 'Q': 'q', 'R': 'r',
        'S': 's', 'T': 't', 'U': 'u', 'V': 'v', 'W': 'w', 'X': 'x', 'Y': 'y', 'Z': 'z'
    }
    result = ''
    for char in word:
        if char in char_map:
            result += char_map[char]
        else:
            result += char
    return result

def lowercase_words(word_list):
    return [manual_lowercase(word) for word in word_list]

stop_words = {
    "is", "a", "the", "for", "this", "an", "and", "in", "of", "on",
    "to", "its", "by", "has", "how", "we", "it", "with", "due", "since", "over", "from"
}

def remove_stop_words(word_list, stop_words):
    return [word for word in word_list if word not in stop_words]

# The manual stemmer and related functions are assumed to be defined above
# I will include manual_stem and apply_stemming only here for brevity:

protected = {
    "universe", "university", "police", "policy",
    "analysis", "analyst", "history", "historic",
    "spring", "king", "thing", "ceiling", "building", "morning", "evening",
    "composed", "supposed", "exposed", "proposed",
    "routing", "housing", "painting", "booking",
    "bringing",
    "news", "species", "series", "means", "physics", "mathematics",
    "economics", "ethics", "politics", "practice", "justice", "notice",
    "service", "advice", "basis", "crisis", "thesis", "oasis",
    "catalyst", "specialist", "specific", "traffic", "arctic",
    "logic", "magic"
}

irregular_verbs = {
    "go": {"past": ["went"], "past_participle": ["gone"]},
    "buy": {"past": ["bought"], "past_participle": ["bought"]},
    "bring": {"past": ["brought"], "past_participle": ["brought"]},
    "run": {"past": ["ran"], "past_participle": ["run"]},
    "begin": {"past": ["began"], "past_participle": ["begun"]},
    "drink": {"past": ["drank"], "past_participle": ["drunk"]},
    "eat": {"past": ["ate"], "past_participle": ["eaten"]},
    "give": {"past": ["gave"], "past_participle": ["given"]},
    "take": {"past": ["took"], "past_participle": ["taken"]},
    "write": {"past": ["wrote"], "past_participle": ["written"]},
    "see": {"past": ["saw"], "past_participle": ["seen"]},
    "do": {"past": ["did"], "past_participle": ["done"]},
    "have": {"past": ["had"], "past_participle": ["had"]},
    "make": {"past": ["made"], "past_participle": ["made"]},
    "say": {"past": ["said"], "past_participle": ["said"]},
    "find": {"past": ["found"], "past_participle": ["found"]},
    "think": {"past": ["thought"], "past_participle": ["thought"]},
    "keep": {"past": ["kept"], "past_participle": ["kept"]},
    "feel": {"past": ["felt"], "past_participle": ["felt"]},
    "leave": {"past": ["left"], "past_participle": ["left"]},
    "meet": {"past": ["met"], "past_participle": ["met"]},
    "stand": {"past": ["stood"], "past_participle": ["stood"]},
    "sit": {"past": ["sat"], "past_participle": ["sat"]},
    "speak": {"past": ["spoke"], "past_participle": ["spoken"]},
    "understand": {"past": ["understood"], "past_participle": ["understood"]},
    "win": {"past": ["won"], "past_participle": ["won"]},
    "lose": {"past": ["lost"], "past_participle": ["lost"]},
    "build": {"past": ["built"], "past_participle": ["built"]},
    "pay": {"past": ["paid"], "past_participle": ["paid"]},
    "send": {"past": ["sent"], "past_participle": ["sent"]},
    "sell": {"past": ["sold"], "past_participle": ["sold"]},
    "catch": {"past": ["caught"], "past_participle": ["caught"]},
    "fight": {"past": ["fought"], "past_participle": ["fought"]},
    "teach": {"past": ["taught"], "past_participle": ["taught"]},
    "hold": {"past": ["held"], "past_participle": ["held"]},
    "hear": {"past": ["heard"], "past_participle": ["heard"]},
    "show": {"past": ["showed"], "past_participle": ["shown"]}
}

def ends_with(word, suffix):
    if len(word) < len(suffix):
        return False
    for i in range(1, len(suffix) + 1):
        if word[-i] != suffix[-i]:
            return False
    return True

def manual_stem(word):
    w = word.lower()
    if w in protected:
        return w
    for base, forms in irregular_verbs.items():
        if w in forms["past"] or w in forms["past_participle"]:
            return base
    length = len(w)
    if ends_with(w, "ies") and length > 3:
        w = w[:-3] + "y"
    elif ends_with(w, "es") and not ends_with(w, "ses") and not ends_with(w, "xes"):
        w = w[:-2]
    elif ends_with(w, "s") and length > 3:
        w = w[:-1]
    if ends_with(w, "ied") and length > 4:
        w = w[:-3] + "y"
    elif ends_with(w, "eed") and length > 3:
        w = w[:-1]
    elif ends_with(w, "ed") and length > 3:
        w = w[:-2]
        if not ends_with(w, "e"):
            w += "e"
    if ends_with(w, "ing") and len(w) > 4:
        w = w[:-3]
        if len(w) > 2 and w[-1] == w[-2]:
            w = w[:-1]
        if not ends_with(w, "e") and w not in protected:
            w += "e"
    if ends_with(w, "tion"):
        w = w[:-4] + "te"
    elif ends_with(w, "sion"):
        w = w[:-4] + "se"
    elif ends_with(w, "ment") or ends_with(w, "ness") or ends_with(w, "less"):
        w = w[:-4]
    elif ends_with(w, "able") or ends_with(w, "ible"):
        w = w[:-4]
    elif ends_with(w, "al") and len(w) > 4:
        w = w[:-2]
    elif ends_with(w, "er") and len(w) > 4:
        w = w[:-2]
    elif ends_with(w, "ly") and len(w) > 3:
        w = w[:-2]
    elif ends_with(w, "ous") and len(w) > 4:
        w = w[:-3]
    elif ends_with(w, "ive") and len(w) > 4:
        w = w[:-3]
    elif ends_with(w, "."):
        w = w[:-1]
    return w

def apply_stemming(word_list):
    return [manual_stem(word) for word in word_list]

# ----------- GIRN Workflow -----------

def get_data(file_path):
    """G: Load data and map labels."""
    data = pd.read_csv(file_path)
    data['label'] = data['sentiment'].map({'positive': 1, 'negative': 0})
    return data['review'], data['label']

def preprocess_text(text):
    """Full custom preprocessing pipeline for one document."""
    words = split_text(text)
    words = remove_punctuation(words)
    words = lowercase_words(words)
    words = remove_stop_words(words, stop_words)
    words = apply_stemming(words)
    return ' '.join(words)  # Return cleaned & stemmed text as a string

def input_preprocessing(text_series):
    """Apply custom preprocessing to entire pandas Series."""
    return text_series.apply(preprocess_text)

def run_model(X, y):
    """R: Train-test split, train SVM, predict."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42)
    
    vectorizer = TfidfVectorizer(max_df=0.7)
    X_train_vect = vectorizer.fit_transform(X_train)
    X_test_vect = vectorizer.transform(X_test)
    
    model = LinearSVC()
    model.fit(X_train_vect, y_train)
    
    y_pred = model.predict(X_test_vect)
    return y_test, y_pred

def notify_results(y_test, y_pred):
    """N: Print classification metrics."""
    print("Classification Report:\n", classification_report(y_test, y_pred, target_names=['negative', 'positive']))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Accuracy Score:", accuracy_score(y_test, y_pred))


if __name__ == "__main__":
    file_path = 'IMDB Dataset.csv'
    
    # G: Get data
    X_text, y = get_data(file_path)
    
    # I: Preprocess text with your custom functions
    X_cleaned = input_preprocessing(X_text)
    
    # R: Train model and predict
    y_test, y_pred = run_model(X_cleaned, y)
    
    # N: Notify results
    notify_results(y_test, y_pred)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\DELL\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\DELL\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\DELL\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\DELL\anaconda3\Lib\site-packages

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import